In [1]:
# Import libraries 
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../data/netflix_customer_churn.csv')
df.head()

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


In [2]:
# check data types and missing values

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())




Data types:
customer_id                object
age                         int64
gender                     object
subscription_type          object
watch_hours               float64
last_login_days             int64
region                     object
device                     object
monthly_fee               float64
churned                     int64
payment_method             object
number_of_profiles          int64
avg_watch_time_per_day    float64
favorite_genre             object
dtype: object

Missing values:
customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64


In [8]:
# How does churn rate vary by subscription type?
df.groupby('subscription_type')['churned'].mean() * 100

subscription_type
Basic       61.830223
Premium     43.709392
Standard    45.443499
Name: churned, dtype: float64

In [9]:
# Does age differ between subscription types?
df.groupby('subscription_type')['age'].mean()

subscription_type
Basic       44.223359
Premium     43.719433
Standard    43.599635
Name: age, dtype: float64

In [10]:
# Do churned users watch less and log in less often?
df.groupby('churned')[['watch_hours', 'last_login_days', 'monthly_fee']].mean()

,watch_hours,last_login_days,monthly_fee
churned,,,
0,17.449590,21.771026,14.248350
1,5.918497,38.309344,13.125189


In [11]:
# Does device or region relate to churn?
df.groupby('device')['churned'].mean() * 100

device
Desktop    49.209694
Laptop     51.789264
Mobile     50.498008
TV         49.949648
Tablet     50.000000
Name: churned, dtype: float64

In [12]:
df.groupby('region')['churned'].mean() * 100

region
Africa           48.318804
Asia             50.653983
Europe           51.672434
North America    49.471210
Oceania          50.065359
South America    51.431844
Name: churned, dtype: float64

In [13]:
# Do households with more profiles churn differently?
df.groupby('number_of_profiles')['churned'].mean() * 100

number_of_profiles
1    58.641975
2    57.442557
3    57.847082
4    37.537538
5    40.618956
Name: churned, dtype: float64

In [14]:
# Is the tier effect explained by engagement? Do Basic users watch less?
df.groupby('subscription_type')[['watch_hours', 'last_login_days']].mean()

,watch_hours,last_login_days
subscription_type,,
Basic,11.563528,30.116195
Premium,11.706486,30.084465
Standard,11.677491,30.068651


Basic users have higher churn despite similar average engagement across subscription tiers.
Engagement and subscription tier are both associated with churn. The higher churn rate among Basic users does not appear to be explained by average watch hours or login recency alone.

In [15]:
# Group login recency into bands to find where churn risk changes
df['login_band'] = pd.cut(df['last_login_days'], bins=[-1, 7, 14, 30, 60, float('inf')],
                          labels=['0-7','8-14','15-30','31-60','60+'])

df.groupby('login_band', observed=True)['churned'].agg(['mean','size']).assign(
    churn_rate=lambda d: (d['mean']*100).round(1))[['churn_rate','size']]

,churn_rate,size
login_band,,
0-7,13.4,650
8-14,27.0,559
15-30,32.0,1322
31-60,75.1,2469


Churn rises sharply after 30 days of inactivity: 32% for users inactive 15–30 days and 75% for users inactive 31–60 days.

In [16]:
# Do low watch time and stale login compound, or say the same thing twice?
df['low_watch'] = df['watch_hours'] < df['watch_hours'].median()
df['stale_login'] = df['last_login_days'] > df['last_login_days'].median()

(df.groupby(['low_watch','stale_login'])['churned'].mean()*100).round(1)

low_watch  stale_login
False      False           2.1
           True           54.8
True       False          49.7
           True           95.8
Name: churned, dtype: float64

In [18]:
print("Key findings:")
print("1. Watch hours and days since last login are the strongest signals")
print("   watch_hours correlates -0.48 with churn, last_login_days +0.47")
print("2. Churned users watch 5.9 hours on average vs 17.4 for active users")
print("3. Churned users last logged in 38 days ago vs 22 days")
print("4. Basic plan churns most at 61.8%, vs Standard 45.4% and Premium 43.7%")
print("5. Age shows no linear relationship with churn (-0.004)")
print("6. Basic users have higher churn despite similar average engagement")
print("   across subscription tiers")
print("7. Churn rises sharply after 30 days of inactivity: 32% at 15-30 days,")
print("   75% at 31-60 days")
print("8. The two signals combine: 2.1% churn with neither, ~50% with either,")
print("   95.8% with both")

Key findings:
1. Watch hours and days since last login are the strongest signals
   watch_hours correlates -0.48 with churn, last_login_days +0.47
2. Churned users watch 5.9 hours on average vs 17.4 for active users
3. Churned users last logged in 38 days ago vs 22 days
4. Basic plan churns most at 61.8%, vs Standard 45.4% and Premium 43.7%
5. Age shows no linear relationship with churn (-0.004)
6. Basic users have higher churn despite similar average engagement
   across subscription tiers
7. Churn rises sharply after 30 days of inactivity: 32% at 15-30 days,
   75% at 31-60 days
8. The two signals combine: 2.1% churn with neither, ~50% with either,
   95.8% with both
